In [1]:
import pandas as pd
import numpy as np
import joblib
import json

from pathlib import Path

project = Path.cwd().parent

print("Project:", project)

Project: c:\Users\yashr\OneDrive\Documents\AI CUSTOMER PREDICTION CHURN


Load the original dataset

In [2]:
df = pd.read_csv(
    project / "Data-raw" / "Telco-Customer-Churn-Cleaned.csv"
)

print("Dataset shape:", df.shape)

display(df.head())

Dataset shape: (7043, 21)


,customerID,gender,SeniorCitizen,Partner,Dependents,tenure,PhoneService,MultipleLines,InternetService,OnlineSecurity,...,DeviceProtection,TechSupport,StreamingTV,StreamingMovies,Contract,PaperlessBilling,PaymentMethod,MonthlyCharges,TotalCharges,Churn
0,7590-VHVEG,Female,0,Yes,No,1,No,No phone service,DSL,No,...,No,No,No,No,Month-to-month,Yes,Electronic check,29.85,29.85,No
1,5575-GNVDE,Male,0,No,No,34,Yes,No,DSL,Yes,...,Yes,No,No,No,One year,No,Mailed check,56.95,1889.50,No
2,3668-QPYBK,Male,0,No,No,2,Yes,No,DSL,Yes,...,No,No,No,No,Month-to-month,Yes,Mailed check,53.85,108.15,Yes
3,7795-CFOCW,Male,0,No,No,45,No,No phone service,DSL,Yes,...,Yes,Yes,No,No,One year,No,Bank transfer (automatic),42.30,1840.75,No
4,9237-HQITU,Female,0,No,No,2,Yes,No,Fiber optic,No,...,No,No,No,No,Month-to-month,Yes,Electronic check,70.70,151.65,Yes


Load the optimized AI model

In [3]:
model = joblib.load(
    project / "models" / "optimized_churn_model.pkl"
)

with open(
    project / "models" / "prediction_settings.json"
) as f:
    settings = json.load(f)

threshold = settings["threshold"]

print("Model loaded!")
print("Decision threshold:", threshold)

Model loaded!
Decision threshold: 0.55


Generate churn probabilities

In [4]:
X_all = df.drop(
    columns=["customerID", "Churn"]
)

probabilities = model.predict_proba(
    X_all
)[:, 1]

dashboard_df = df.copy()

dashboard_df["ChurnProbability"] = probabilities

dashboard_df["PredictedChurn"] = np.where(
    probabilities >= threshold,
    "Yes",
    "No"
)

dashboard_df["RiskCategory"] = pd.cut(
    dashboard_df["ChurnProbability"],
    bins=[-0.001, 0.30, 0.60, 1.0],
    labels=["Low", "Medium", "High"]
)

display(
    dashboard_df[
        [
            "customerID",
            "Churn",
            "ChurnProbability",
            "PredictedChurn",
            "RiskCategory"
        ]
    ].head()
)

,customerID,Churn,ChurnProbability,PredictedChurn,RiskCategory
0,7590-VHVEG,No,0.774184,Yes,High
1,5575-GNVDE,No,0.139776,No,Low
2,3668-QPYBK,Yes,0.525407,No,Medium
3,7795-CFOCW,No,0.096286,No,Low
4,9237-HQITU,Yes,0.854317,Yes,High


Save the Power BI dataset

In [5]:
output_folder = project / "Data-processed"

output_folder.mkdir(
    parents=True,
    exist_ok=True
)

output_file = (
    output_folder /
    "PowerBI_Customer_Churn.csv"
)

dashboard_df.to_csv(
    output_file,
    index=False
)

print("File saved:", output_file.exists())
print("Location:", output_file.resolve())

File saved: True
Location: C:\Users\yashr\OneDrive\Documents\AI CUSTOMER PREDICTION CHURN\Data-processed\PowerBI_Customer_Churn.csv
